In [1]:
from pathlib import Path
import json
import gc
import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import (
    f1_score,
    confusion_matrix,
    classification_report,
    balanced_accuracy_score,
    accuracy_score,
)

# =========================================================
# 0) Reproducibility
# =========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# =========================================================
# 1) Paths
# =========================================================
FEATURE_ROOT = Path(r"C:\SeniorProject\Good_Data\NPZ_SPLITS_FILTERED_CAP1H_IMPD\CNN_FEATURES_FOR_BILSTM_BALACCFULL24")
TRAIN_ROOT = FEATURE_ROOT / "TRAIN"
VAL_ROOT   = FEATURE_ROOT / "VAL"
TEST_ROOT  = FEATURE_ROOT / "TEST"

OUT_ROOT = FEATURE_ROOT.parent / "BILSTM_RESULTS_PAPER_STYLE_LDAM_DRW_BALACC_FEATURES24F"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

BEST_CKPT_PATH = OUT_ROOT / "best_model.pt"
LAST_CKPT_PATH = OUT_ROOT / "last_model.pt"
METRICS_JSON   = OUT_ROOT / "final_metrics.json"
CLS_REPORT_TXT = OUT_ROOT / "test_classification_report.txt"
CM_NPY         = OUT_ROOT / "test_confusion_matrix.npy"
PRED_NPZ       = OUT_ROOT / "test_predictions.npz"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(torch.cuda.current_device())
    print("GPU:", props.name)
    print("VRAM (GB):", round(props.total_memory / (1024**3), 2))
print("Torch:", torch.__version__)

# =========================================================
# 2) Config
# =========================================================
NUM_CLASSES = 3
CLASS_NAMES = {0: "Normal", 1: "Preictal", 2: "Ictal"}

# ---------------------------------------------------------
# Sequence setup
# ---------------------------------------------------------
SEQ_LEN = 8
SEQ_STRIDE = 1

# ---------------------------------------------------------
# Training
# ---------------------------------------------------------
BATCH_SIZE = 128
MAX_EPOCHS = 90
MIN_EPOCHS = 40
PATIENCE = 10

LR = 1e-3
CLIP_NORM = 1.0

# fixed step schedule
LR_MILESTONES = [30, 60]
LR_GAMMA = 0.1

# ---------------------------------------------------------
# Model
# ---------------------------------------------------------
INPUT_DIM = 256
HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.2
INPUT_DROPOUT = 0.1

# ---------------------------------------------------------
# LDAM-DRW
# ---------------------------------------------------------
LDAM_C = 0.5
DRW_START_EPOCH = 21   # صار أبكر بدل 61

NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

# =========================================================
# 3) Helpers
# =========================================================
def load_feature_run(path: Path):
    npz = np.load(path, allow_pickle=True)
    F_arr = np.asarray(npz["F"], dtype=np.float32)
    y_arr = np.asarray(npz["y"], dtype=np.int64)
    win_idx = np.asarray(npz["win_idx"], dtype=np.int32)
    base = str(npz["base"][0]) if "base" in npz else path.stem
    return F_arr, y_arr, win_idx, base

def compute_train_feature_stats(train_root: Path):
    sum_vec = None
    sumsq_vec = None
    total_n = 0

    files = sorted(train_root.glob("*.npz"))
    if len(files) == 0:
        raise RuntimeError(f"No feature files found in {train_root}")

    for p in files:
        F_arr, _, _, _ = load_feature_run(p)
        if F_arr.ndim != 2 or F_arr.shape[0] == 0:
            continue

        if sum_vec is None:
            sum_vec = F_arr.sum(axis=0, dtype=np.float64)
            sumsq_vec = (F_arr ** 2).sum(axis=0, dtype=np.float64)
        else:
            sum_vec += F_arr.sum(axis=0, dtype=np.float64)
            sumsq_vec += (F_arr ** 2).sum(axis=0, dtype=np.float64)

        total_n += F_arr.shape[0]

    if total_n == 0:
        raise RuntimeError("No valid TRAIN features found.")

    mean = sum_vec / total_n
    var = (sumsq_vec / total_n) - (mean ** 2)
    var = np.maximum(var, 1e-8)
    std = np.sqrt(var)

    return mean.astype(np.float32), std.astype(np.float32)

def compute_sequence_counts(root: Path, seq_len: int, seq_stride: int):
    counts = np.zeros(NUM_CLASSES, dtype=np.int64)

    for p in sorted(root.glob("*.npz")):
        _, y_arr, _, _ = load_feature_run(p)
        n = len(y_arr)
        if n < seq_len:
            continue

        for s in range(0, n - seq_len + 1, seq_stride):
            label = int(y_arr[s + seq_len - 1])
            counts[label] += 1

    counts = np.maximum(counts, 1)
    return counts

def make_paper_drw_weights(class_counts: np.ndarray) -> np.ndarray:
    """
    Inverse frequency, then normalized for stable scale.
    """
    counts = class_counts.astype(np.float64)
    weights = 1.0 / np.maximum(counts, 1.0)
    weights = weights / weights.mean()
    return weights.astype(np.float32)

def make_sequence_sample_weights(dataset, class_counts: np.ndarray) -> np.ndarray:
    """
    Weight per sequence sample for WeightedRandomSampler.
    Each sequence gets weight based on its sequence label.
    """
    class_weights = 1.0 / np.maximum(class_counts.astype(np.float64), 1.0)
    class_weights = class_weights / class_weights.sum()

    sample_weights = np.zeros(len(dataset.samples), dtype=np.float64)

    for i, (p, s) in enumerate(dataset.samples):
        _, y_arr, _, _ = dataset.cache[p]
        label = int(y_arr[s + dataset.seq_len - 1])
        sample_weights[i] = class_weights[label]

    return sample_weights.astype(np.float64)

# =========================================================
# 4) Dataset
# =========================================================
class SequenceFeatureDataset(Dataset):
    def __init__(self, root: Path, seq_len: int, seq_stride: int,
                 feat_mean: np.ndarray, feat_std: np.ndarray):
        self.root = Path(root)
        self.seq_len = int(seq_len)
        self.seq_stride = int(seq_stride)
        self.feat_mean = feat_mean.astype(np.float32)
        self.feat_std = feat_std.astype(np.float32)

        self.run_files = sorted(self.root.glob("*.npz"))
        if len(self.run_files) == 0:
            raise RuntimeError(f"No feature files found in {self.root}")

        self.samples = []
        self.cache = {}

        for p in self.run_files:
            F_arr, y_arr, win_idx, base = load_feature_run(p)

            if F_arr.ndim != 2:
                continue
            if len(y_arr) != F_arr.shape[0] or len(win_idx) != F_arr.shape[0]:
                continue

            n = F_arr.shape[0]
            if n < self.seq_len:
                continue

            self.cache[str(p)] = (F_arr, y_arr, win_idx, base)

            for s in range(0, n - self.seq_len + 1, self.seq_stride):
                self.samples.append((str(p), s))

        if len(self.samples) == 0:
            raise RuntimeError(f"No valid sequences found in {self.root}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        p, s = self.samples[idx]
        F_arr, y_arr, win_idx, base = self.cache[p]

        e = s + self.seq_len
        X_seq = F_arr[s:e]
        X_seq = (X_seq - self.feat_mean) / self.feat_std
        X_seq = np.nan_to_num(X_seq, nan=0.0, posinf=0.0, neginf=0.0)

        label = int(y_arr[e - 1])
        end_wi = int(win_idx[e - 1])

        return (
            torch.from_numpy(X_seq.astype(np.float32)),
            torch.tensor(label, dtype=torch.long),
            base,
            torch.tensor(end_wi, dtype=torch.long),
        )

def collate_seq(batch):
    Xs, ys, bases, end_wis = [], [], [], []
    for X, y, base, wi in batch:
        Xs.append(X)
        ys.append(y)
        bases.append(base)
        end_wis.append(wi)

    return (
        torch.stack(Xs, dim=0),
        torch.stack(ys, dim=0),
        bases,
        torch.stack(end_wis, dim=0)
    )

# =========================================================
# 5) LDAM Loss
# =========================================================
class LDAMLoss(nn.Module):
    """
    Delta_j = C / n_j^(1/4)
    """
    def __init__(self, class_counts, C=0.5, weight=None, s=30):
        super().__init__()

        counts = np.asarray(class_counts, dtype=np.float32)
        counts = np.maximum(counts, 1.0)

        m_list = C / np.power(counts, 0.25)
        m_list = torch.tensor(m_list, dtype=torch.float32)

        self.register_buffer("m_list", m_list)
        self.s = float(s)

        if weight is not None:
            weight = torch.as_tensor(weight, dtype=torch.float32)
            self.register_buffer("weight", weight)
        else:
            self.weight = None

    def set_weight(self, weight):
        if weight is None:
            self.weight = None
        else:
            weight = torch.as_tensor(weight, dtype=torch.float32, device=self.m_list.device)
            self.weight = weight

    def forward(self, logits, target):
        target = target.long()
        index = torch.zeros_like(logits, dtype=torch.bool)
        index.scatter_(1, target.view(-1, 1), True)

        batch_m = self.m_list[target].view(-1, 1)
        logits_m = logits - batch_m * index.float()

        return F.cross_entropy(self.s * logits_m, target, weight=self.weight)

# =========================================================
# 6) Model
# =========================================================
class FinalBiLSTM(nn.Module):
    def __init__(self, input_dim=256, hidden_size=128, num_layers=2, dropout=0.2, num_classes=3, input_dropout=0.1):
        super().__init__()

        self.input_norm = nn.LayerNorm(input_dim)
        self.input_dropout = nn.Dropout(input_dropout)

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        x = self.input_norm(x)
        x = self.input_dropout(x)
        h, _ = self.lstm(x)
        last_h = h[:, -1, :]
        last_h = self.dropout(last_h)
        logits = self.classifier(last_h)
        return logits

# =========================================================
# 7) Train / Eval
# =========================================================
@torch.no_grad()
def evaluate(model, loader, device, loss_fn):
    model.eval()

    total_loss = 0.0
    n = 0
    all_y = []
    all_p = []
    all_probs = []
    all_bases = []
    all_end_wis = []

    for X, y, bases, end_wis in loader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(X)
        loss = loss_fn(logits, y)

        if not torch.isfinite(loss):
            raise RuntimeError("Validation/Test loss became NaN or Inf.")

        probs = torch.softmax(logits, dim=1)
        pred = torch.argmax(probs, dim=1)

        total_loss += float(loss.item()) * y.size(0)
        n += y.size(0)

        all_y.append(y.detach().cpu().numpy())
        all_p.append(pred.detach().cpu().numpy())
        all_probs.append(probs.detach().cpu().numpy())
        all_bases.extend(bases)
        all_end_wis.append(end_wis.detach().cpu().numpy())

    y_true = np.concatenate(all_y, axis=0)
    y_pred = np.concatenate(all_p, axis=0)
    probs = np.concatenate(all_probs, axis=0)
    end_wis = np.concatenate(all_end_wis, axis=0)

    avg_loss = total_loss / max(1, n)

    return {
        "loss": avg_loss,
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "y_true": y_true,
        "y_pred": y_pred,
        "probs": probs,
        "bases": np.asarray(all_bases, dtype=object),
        "end_wis": end_wis
    }

def train_one_epoch(model, loader, optimizer, device, loss_fn, clip_norm=1.0):
    model.train()

    total_loss = 0.0
    n = 0

    for X, y, _, _ in loader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(X)
        loss = loss_fn(logits, y)

        if not torch.isfinite(loss):
            raise RuntimeError("Training loss became NaN or Inf.")

        loss.backward()

        if clip_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)

        optimizer.step()

        total_loss += float(loss.item()) * y.size(0)
        n += y.size(0)

    return total_loss / max(1, n)

# =========================================================
# 8) Main
# =========================================================
def main():
    feat_mean, feat_std = compute_train_feature_stats(TRAIN_ROOT)
    print("Feature dim:", len(feat_mean))

    train_ds = SequenceFeatureDataset(TRAIN_ROOT, SEQ_LEN, SEQ_STRIDE, feat_mean, feat_std)
    val_ds   = SequenceFeatureDataset(VAL_ROOT,   SEQ_LEN, SEQ_STRIDE, feat_mean, feat_std)
    test_ds  = SequenceFeatureDataset(TEST_ROOT,  SEQ_LEN, SEQ_STRIDE, feat_mean, feat_std)

    class_counts = compute_sequence_counts(TRAIN_ROOT, SEQ_LEN, SEQ_STRIDE)
    print("TRAIN sequence label counts:", {i: int(class_counts[i]) for i in range(NUM_CLASSES)})

    drw_weights = make_paper_drw_weights(class_counts)
    print("Normalized DRW class weights:", drw_weights.tolist())

    # -----------------------------------------------------
    # WeightedRandomSampler
    # -----------------------------------------------------
    train_sample_weights = make_sequence_sample_weights(train_ds, class_counts)
    train_sampler = WeightedRandomSampler(
        weights=torch.as_tensor(train_sample_weights, dtype=torch.double),
        num_samples=len(train_sample_weights),
        replacement=True
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        sampler=train_sampler,   # بدل shuffle=True
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        collate_fn=collate_seq
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        collate_fn=collate_seq
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        collate_fn=collate_seq
    )

    print("Train sequences:", len(train_ds))
    print("Val sequences  :", len(val_ds))
    print("Test sequences :", len(test_ds))
    print("Sampler enabled: WeightedRandomSampler")

    loss_fn = LDAMLoss(
        class_counts=class_counts,
        C=LDAM_C,
        weight=None,
        s=30
    ).to(DEVICE)

    model = FinalBiLSTM(
        input_dim=INPUT_DIM,
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        num_classes=NUM_CLASSES,
        input_dropout=INPUT_DROPOUT
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=LR_MILESTONES,
        gamma=LR_GAMMA
    )

    X0, _, _, _ = next(iter(train_loader))
    X0 = X0.to(DEVICE)
    with torch.no_grad():
        logits0 = model(X0)
    print("Sanity logits min/max:", logits0.min().item(), logits0.max().item())
    print("Sanity logits finite :", torch.isfinite(logits0).all().item())

    # best model tracking
    best_macro_f1 = -1.0
    best_bal_acc = -1.0
    best_val_loss = float("inf")
    best_epoch = -1
    bad_epochs = 0

    # tolerance to avoid saving for tiny noisy changes
    MACRO_EPS = 1e-3
    BAL_EPS = 1e-4
    LOSS_EPS = 1e-4

    print("\n[1/2] Training LDAM-DRW + WeightedRandomSampler...")
    for epoch in range(1, MAX_EPOCHS + 1):
        if epoch == DRW_START_EPOCH:
            print(f"--> DRW activated at epoch {epoch}")
            loss_fn.set_weight(drw_weights)

        current_weight_status = "ON" if loss_fn.weight is not None else "OFF"

        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            device=DEVICE,
            loss_fn=loss_fn,
            clip_norm=CLIP_NORM
        )

        val_out = evaluate(
            model=model,
            loader=val_loader,
            device=DEVICE,
            loss_fn=loss_fn
        )

        val_loss = val_out["loss"]
        val_macro = val_out["macro_f1"]
        val_bal_acc = val_out["balanced_acc"]
        val_weighted = val_out["weighted_f1"]

        lr_now = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch {epoch:03d} | "
            f"DRW {current_weight_status} | "
            f"lr {lr_now:.2e} | "
            f"train_loss {train_loss:.4f} | "
            f"val_loss {val_loss:.4f} | "
            f"val_bal_acc {val_bal_acc:.4f} | "
            f"val_macro_f1 {val_macro:.4f} | "
            f"val_weighted_f1 {val_weighted:.4f}"
        )

        torch.save({
            "epoch": epoch,
            "model": model.state_dict(),
            "optim": optimizer.state_dict(),
        }, LAST_CKPT_PATH)

        # -------------------------------------------------
        # Best model logic:
        # 1) macro_f1 first
        # 2) then balanced_acc
        # 3) then val_loss
        # with tolerance to reduce noisy saves
        # -------------------------------------------------
        improved = False

        if val_macro > best_macro_f1 + MACRO_EPS:
            improved = True
        elif abs(val_macro - best_macro_f1) <= MACRO_EPS:
            if val_bal_acc > best_bal_acc + BAL_EPS:
                improved = True
            elif abs(val_bal_acc - best_bal_acc) <= BAL_EPS:
                if val_loss < best_val_loss - LOSS_EPS:
                    improved = True

        if improved:
            best_macro_f1 = val_macro
            best_bal_acc = val_bal_acc
            best_val_loss = val_loss
            best_epoch = epoch
            bad_epochs = 0

            torch.save({
                "epoch": epoch,
                "model": model.state_dict(),
                "optim": optimizer.state_dict(),
                "best_macro_f1": best_macro_f1,
                "best_bal_acc": best_bal_acc,
                "best_val_loss": best_val_loss,
                "seq_len": SEQ_LEN,
                "input_dim": INPUT_DIM,
                "hidden_size": HIDDEN_SIZE,
                "num_layers": NUM_LAYERS,
                "dropout": DROPOUT,
                "input_dropout": INPUT_DROPOUT,
                "lr": LR,
                "ldam_C": LDAM_C,
                "drw_start_epoch": DRW_START_EPOCH,
                "lr_milestones": LR_MILESTONES,
                "lr_gamma": LR_GAMMA
            }, BEST_CKPT_PATH)

            print(
                f"✅ Saved BEST checkpoint "
                f"(macro_f1={best_macro_f1:.4f}, bal_acc={best_bal_acc:.4f}, val_loss={best_val_loss:.4f})"
            )
        else:
            bad_epochs += 1

        early_stop_ready = epoch >= max(MIN_EPOCHS, DRW_START_EPOCH + 10)

        if early_stop_ready and bad_epochs >= PATIENCE:
            print(f"🛑 Early stopping at epoch {epoch}. Best epoch={best_epoch}")
            break

        scheduler.step()

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print("\n[2/2] Final TEST evaluation...")
    ckpt = torch.load(BEST_CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])

    if ckpt["epoch"] >= DRW_START_EPOCH:
        loss_fn.set_weight(drw_weights)
    else:
        loss_fn.set_weight(None)

    test_out = evaluate(model, test_loader, DEVICE, loss_fn)

    y_true = test_out["y_true"]
    y_pred = test_out["y_pred"]
    probs  = test_out["probs"]
    bases  = test_out["bases"]
    end_wis = test_out["end_wis"]

    test_loss = test_out["loss"]
    acc = accuracy_score(y_true, y_pred)
    bal_acc = test_out["balanced_acc"]
    macro_f1 = test_out["macro_f1"]
    weighted_f1 = test_out["weighted_f1"]

    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(y_true, y_pred, digits=4, zero_division=0)

    print("\n===== FINAL TEST RESULTS =====")
    print("Best epoch             :", ckpt["epoch"])
    print("Test loss              :", test_loss)
    print("Test accuracy          :", acc)
    print("Test balanced accuracy :", bal_acc)
    print("Test macro_f1          :", macro_f1)
    print("Test weighted_f1       :", weighted_f1)
    print("Confusion matrix:\n", cm)
    print("\nClassification report:\n", report)

    with open(CLS_REPORT_TXT, "w", encoding="utf-8") as f:
        f.write(report)

    np.save(CM_NPY, cm)

    np.savez_compressed(
        PRED_NPZ,
        y_true=y_true.astype(np.int64),
        y_pred=y_pred.astype(np.int64),
        probs=probs.astype(np.float32),
        bases=bases,
        end_wis=end_wis.astype(np.int64),
    )

    metrics = {
        "best_epoch": int(ckpt["epoch"]),
        "test_loss": float(test_loss),
        "test_accuracy": float(acc),
        "test_balanced_accuracy": float(bal_acc),
        "test_macro_f1": float(macro_f1),
        "test_weighted_f1": float(weighted_f1),
        "best_val_macro_f1": float(ckpt.get("best_macro_f1", -1.0)),
        "best_val_bal_acc": float(ckpt.get("best_bal_acc", -1.0)),
        "best_val_loss": float(ckpt.get("best_val_loss", -1.0)),
        "seq_len": int(SEQ_LEN),
        "input_dim": int(INPUT_DIM),
        "hidden_size": int(HIDDEN_SIZE),
        "num_layers": int(NUM_LAYERS),
        "dropout": float(DROPOUT),
        "input_dropout": float(INPUT_DROPOUT),
        "lr": float(LR),
        "ldam_C": float(LDAM_C),
        "drw_start_epoch": int(DRW_START_EPOCH),
        "lr_milestones": LR_MILESTONES,
        "lr_gamma": float(LR_GAMMA),
    }

    with open(METRICS_JSON, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    print("\n✅ All done.")
    print("Results folder:", OUT_ROOT)

if __name__ == "__main__":
    main()

CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
VRAM (GB): 7.96
Torch: 2.10.0+cu128
Feature dim: 256
TRAIN sequence label counts: {0: 1704691, 1: 127548, 2: 11053}
Normalized DRW class weights: [0.017794238403439522, 0.23782166838645935, 2.7443840503692627]
Train sequences: 1843292
Val sequences  : 154213
Test sequences : 29990589
Sampler enabled: WeightedRandomSampler
Sanity logits min/max: -0.1757674217224121 0.1313457190990448
Sanity logits finite : True

[1/2] Training LDAM-DRW + WeightedRandomSampler...
Epoch 001 | DRW OFF | lr 1.00e-03 | train_loss 0.4376 | val_loss 1.4049 | val_bal_acc 0.4226 | val_macro_f1 0.3488 | val_weighted_f1 0.7549
✅ Saved BEST checkpoint (macro_f1=0.3488, bal_acc=0.4226, val_loss=1.4049)
Epoch 002 | DRW OFF | lr 1.00e-03 | train_loss 0.2088 | val_loss 1.3825 | val_bal_acc 0.4007 | val_macro_f1 0.3515 | val_weighted_f1 0.7772
✅ Saved BEST checkpoint (macro_f1=0.3515, bal_acc=0.4007, val_loss=1.3825)
Epoch 003 | DRW OFF | lr 1.00e-03 | train_l

In [2]:
from pathlib import Path
import json
import numpy as np
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

# =========================================================
# 1) Paths
# =========================================================
OUT_ROOT = Path(r"C:\SeniorProject\Good_Data\NPZ_SPLITS_FILTERED_CAP1H_IMPD\BILSTM_RESULTS_PAPER_STYLE_LDAM_DRW_BALACC_FEATURES24F")

METRICS_JSON = OUT_ROOT / "final_metrics.json"
PRED_NPZ     = OUT_ROOT / "test_predictions.npz"
CM_NPY       = OUT_ROOT / "test_confusion_matrix.npy"

CLASS_NAMES = {
    0: "Normal",
    1: "Preictal",
    2: "Ictal"
}

# =========================================================
# 2) Load saved results
# =========================================================
with open(METRICS_JSON, "r", encoding="utf-8") as f:
    saved_metrics = json.load(f)

pred_data = np.load(PRED_NPZ, allow_pickle=True)
y_true = pred_data["y_true"]
y_pred = pred_data["y_pred"]
probs  = pred_data["probs"]
bases  = pred_data["bases"]
end_wis = pred_data["end_wis"]

cm = np.load(CM_NPY)

# =========================================================
# 3) General metrics
# =========================================================
print("=" * 70)
print("GENERAL METRICS")
print("=" * 70)

print(f"Best epoch             : {saved_metrics.get('best_epoch')}")
print(f"Test loss              : {saved_metrics.get('test_loss'):.6f}")
print(f"Test accuracy          : {saved_metrics.get('test_accuracy'):.6f}")
print(f"Test balanced accuracy : {saved_metrics.get('test_balanced_accuracy'):.6f}")
print(f"Test macro F1          : {saved_metrics.get('test_macro_f1'):.6f}")
print(f"Test weighted F1       : {saved_metrics.get('test_weighted_f1'):.6f}")

print("\nExtra recomputed metrics:")
print(f"Accuracy               : {accuracy_score(y_true, y_pred):.6f}")
print(f"Balanced accuracy      : {balanced_accuracy_score(y_true, y_pred):.6f}")
print(f"Macro precision        : {precision_score(y_true, y_pred, average='macro', zero_division=0):.6f}")
print(f"Macro recall           : {recall_score(y_true, y_pred, average='macro', zero_division=0):.6f}")
print(f"Macro F1               : {f1_score(y_true, y_pred, average='macro', zero_division=0):.6f}")
print(f"Weighted precision     : {precision_score(y_true, y_pred, average='weighted', zero_division=0):.6f}")
print(f"Weighted recall        : {recall_score(y_true, y_pred, average='weighted', zero_division=0):.6f}")
print(f"Weighted F1            : {f1_score(y_true, y_pred, average='weighted', zero_division=0):.6f}")

# =========================================================
# 4) Confusion matrix
# =========================================================
print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)
print("Rows = True labels, Columns = Predicted labels\n")

header = " " * 12 + "".join([f"{CLASS_NAMES[i]:>12}" for i in range(len(CLASS_NAMES))])
print(header)

for i in range(len(CLASS_NAMES)):
    row_str = f"{CLASS_NAMES[i]:>12}" + "".join([f"{cm[i, j]:>12d}" for j in range(len(CLASS_NAMES))])
    print(row_str)

# =========================================================
# 5) Classification report
# =========================================================
print("\n" + "=" * 70)
print("SKLEARN CLASSIFICATION REPORT")
print("=" * 70)

report = classification_report(
    y_true,
    y_pred,
    target_names=[CLASS_NAMES[i] for i in range(len(CLASS_NAMES))],
    digits=6,
    zero_division=0
)
print(report)

# =========================================================
# 6) Per-class detailed metrics: TP, FP, FN, TN, specificity...
# =========================================================
print("\n" + "=" * 70)
print("PER-CLASS DETAILED METRICS")
print("=" * 70)

num_classes = len(CLASS_NAMES)
total_samples = len(y_true)

for c in range(num_classes):
    tp = cm[c, c]
    fp = cm[:, c].sum() - tp
    fn = cm[c, :].sum() - tp
    tn = cm.sum() - (tp + fp + fn)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0   # sensitivity
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    class_acc = (tp + tn) / total_samples if total_samples > 0 else 0.0

    support_true = (y_true == c).sum()
    support_pred = (y_pred == c).sum()

    print(f"\nClass {c} -> {CLASS_NAMES[c]}")
    print("-" * 50)
    print(f"Support (true)         : {support_true}")
    print(f"Predicted as class     : {support_pred}")
    print(f"TP                     : {tp}")
    print(f"FP                     : {fp}")
    print(f"FN                     : {fn}")
    print(f"TN                     : {tn}")
    print(f"Precision              : {precision:.6f}")
    print(f"Recall / Sensitivity   : {recall:.6f}")
    print(f"Specificity            : {specificity:.6f}")
    print(f"NPV                    : {npv:.6f}")
    print(f"F1-score               : {f1:.6f}")
    print(f"One-vs-Rest Accuracy   : {class_acc:.6f}")

# =========================================================
# 7) Distribution of true and predicted labels
# =========================================================
print("\n" + "=" * 70)
print("LABEL DISTRIBUTIONS")
print("=" * 70)

for c in range(num_classes):
    n_true = int((y_true == c).sum())
    n_pred = int((y_pred == c).sum())
    print(f"{CLASS_NAMES[c]:<10} | true: {n_true:<8} | pred: {n_pred:<8}")

# =========================================================
# 8) Show some prediction examples
# =========================================================
print("\n" + "=" * 70)
print("FIRST 20 PREDICTIONS")
print("=" * 70)

n_show = min(20, len(y_true))
for i in range(n_show):
    true_name = CLASS_NAMES[int(y_true[i])]
    pred_name = CLASS_NAMES[int(y_pred[i])]
    prob_str = ", ".join([f"{CLASS_NAMES[j]}={probs[i][j]:.4f}" for j in range(num_classes)])
    print(
        f"[{i:03d}] base={bases[i]} | end_wi={int(end_wis[i])} | "
        f"true={true_name} | pred={pred_name} | probs: {prob_str}"
    )

GENERAL METRICS
Best epoch             : 8
Test loss              : 0.154179
Test accuracy          : 0.962052
Test balanced accuracy : 0.977364
Test macro F1          : 0.875111
Test weighted F1       : 0.964687

Extra recomputed metrics:
Accuracy               : 0.962052
Balanced accuracy      : 0.977364
Macro precision        : 0.803976
Macro recall           : 0.977364
Macro F1               : 0.875111
Weighted precision     : 0.971876
Weighted recall        : 0.962052
Weighted F1            : 0.964687

CONFUSION MATRIX
Rows = True labels, Columns = Predicted labels

                  Normal    Preictal       Ictal
      Normal    26073680     1002694       78074
    Preictal       36573     2556978       19225
       Ictal        1088         420      221857

SKLEARN CLASSIFICATION REPORT
              precision    recall  f1-score   support

      Normal   0.998558  0.960199  0.979003  27154448
    Preictal   0.718234  0.978644  0.828457   2612776
       Ictal   0.695137  0.99324

In [ ]:
from pathlib import Path
import numpy as np

OUT_ROOT = Path(r"C:\SeniorProject\Good_Data\NPZ_SPLITS_FILTERED_CAP1H_IMPD\BILSTM_RESULTS_PAPER_STYLE_LDAM_DRW_BALACC_FEATURES24F")

cm = np.load(OUT_ROOT / "test_confusion_matrix.npy")

CLASS_NAMES = {
    0: "Normal",
    1: "Preictal",
    2: "Ictal"
}

print("=" * 80)
print(f"{'Class':<12} {'TP':>8} {'FP':>8} {'FN':>8} {'TN':>8} {'TPR':>10} {'FPR':>10} {'FNR':>10} {'TNR':>10} {'Prec':>10}")
print("=" * 80)

for c in range(len(CLASS_NAMES)):
    tp = cm[c, c]
    fp = cm[:, c].sum() - tp
    fn = cm[c, :].sum() - tp
    tn = cm.sum() - tp - fp - fn

    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0   # Recall / Sensitivity
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0   # False Positive Rate
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0   # False Negative Rate
    tnr = tn / (tn + fp) if (tn + fp) > 0 else 0.0   # Specificity
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0  # Precision

    print(f"{CLASS_NAMES[c]:<12} {tp:>8d} {fp:>8d} {fn:>8d} {tn:>8d} {tpr:>10.4f} {fpr:>10.4f} {fnr:>10.4f} {tnr:>10.4f} {prec:>10.4f}")

Class              TP       FP       FN       TN        TPR        FPR        FNR        TNR       Prec
Normal       26073680    37661  1080768  2798480     0.9602     0.0133     0.0398     0.9867     0.9986
Preictal      2556978  1003114    55798 26374699     0.9786     0.0366     0.0214     0.9634     0.7182
Ictal          221857    97299     1508 29669925     0.9932     0.0033     0.0068     0.9967     0.6951
